# Semana 12 · Buscar el sesgo

**Principios de Inteligencia Artificial · CORHUILA**

**No tienes que escribir código.** Ejecuta las celdas en orden, de arriba abajo: haz clic en una celda y presiona **Shift + Enter**, o el botón ▶ que aparece a su izquierda.

Cuando haya algo que cambiar, estará marcado con **⬅️ CAMBIA ESTE NÚMERO**. No toques nada más: si algo se daña, en el menú *Entorno de ejecución → Reiniciar y ejecutar todo* vuelve a dejarlo como estaba.

Vas a hacer el trabajo de un auditor: tomar un modelo que decide créditos y averiguar si trata igual a dos grupos de personas, **A** y **B**.

In [ ]:
# 1. Traer las herramientas (se ejecuta una sola vez)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
print("Listo: herramientas cargadas.")

## 1. Los datos

Son 4.000 solicitudes de crédito **inventadas**. De cada persona sabemos su ingreso mensual (en millones), cuántas deudas tiene, en qué barrio vive (del 1 al 5) y a qué grupo pertenece.

Hay dos columnas de respuesta, y la diferencia entre ellas es el corazón de la práctica:

- **aprobado**: lo que el banco decidió **en el pasado**. Con esto se entrena el modelo, como pasa en la vida real.
- **pagaría**: si la persona de verdad habría pagado. En la vida real esto casi nunca se sabe; aquí lo sabemos porque los datos son inventados, y eso nos deja **medir la injusticia**.

Un detalle importante: en estos datos, los dos grupos pagan igual de bien. Pero en el pasado, el banco le negó créditos al grupo B con más frecuencia.

In [ ]:
# 2. Crear las 4.000 solicitudes (siempre salen las mismas)
rng = np.random.default_rng(3)
n = 4000
grupo = rng.choice(["A", "B"], n)
# el grupo B vive sobre todo en los barrios 4 y 5
barrio = np.where(grupo == "A",
                  rng.choice([1, 2, 3, 4, 5], n, p=[.35, .35, .2, .05, .05]),
                  rng.choice([1, 2, 3, 4, 5], n, p=[.05, .05, .2, .35, .35]))
ingreso = rng.uniform(1, 8, n).round(1)
deudas = rng.integers(0, 4, n)
pagaria = (ingreso - 1.2 * deudas + rng.normal(0, 1, n)) > 2.5   # igual para los dos grupos
# el banco, en el pasado, le nego el credito a la mitad de los buenos pagadores del grupo B
aprobado = pagaria & ~((grupo == "B") & (rng.random(n) < 0.5))

datos = pd.DataFrame({"grupo": grupo, "barrio": barrio, "ingreso": ingreso,
                      "deudas": deudas, "pagaría": pagaria, "aprobado": aprobado})
datos["es_del_grupo_B"] = (datos["grupo"] == "B").astype(int)
datos.head()

## 2. Entrenar el modelo con las decisiones del pasado

Primero dejamos que el modelo vea **todas** las columnas, incluida la del grupo. Luego vas a probar qué pasa si la borras.

In [ ]:
# 3. Entrenar y medir la exactitud general
usar_columna_grupo = 1   # ⬅️ CAMBIA ESTE NÚMERO: 1 = la usa, 0 = la borra

columnas = ["ingreso", "deudas", "barrio"]
if usar_columna_grupo == 1:
    columnas.append("es_del_grupo_B")

entrena, prueba = train_test_split(datos, test_size=0.3, random_state=0)
modelo = DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, random_state=0)
modelo.fit(entrena[columnas], entrena["aprobado"])
prueba = prueba.assign(decision=modelo.predict(prueba[columnas]))

exactitud = (prueba["decision"] == prueba["aprobado"]).mean()
print("Columnas que ve el modelo:", columnas)
print(f"Exactitud general: {exactitud:.0%}")

## 3. La prueba del auditor: los errores por grupo

La exactitud general se ve bien. Pero la pregunta del auditor es otra: **entre las personas que sí habrían pagado, ¿a cuántas les negó el crédito el modelo, en cada grupo?**

In [ ]:
# 4. Rechazos injustos por grupo
buenos = prueba[prueba["pagaría"]]
injusto = {}
for g in ["A", "B"]:
    del_grupo = buenos[buenos["grupo"] == g]
    injusto[g] = 1 - del_grupo["decision"].mean()
    print(f"Grupo {g}: {len(del_grupo)} buenos pagadores, se le negó el crédito al {injusto[g]:.0%}")

plt.bar(list(injusto), [100 * v for v in injusto.values()], color=["#0e7c7b", "#e07b39"])
plt.ylabel("% de buenos pagadores rechazados")
plt.title("Rechazos injustos por grupo")
plt.show()

## 4. Lo que tienes que responder

1. Con **usar_columna_grupo = 1**, anota la exactitud general y el porcentaje de rechazos injustos de cada grupo. ¿Cuántas veces más rechazos injustos sufre el grupo B?
2. Si solo hubieras mirado la exactitud general, ¿habrías notado el problema?
3. Cambia a **usar_columna_grupo = 0** y usa *Entorno de ejecución → Ejecutar todas*. ¿Desaparece la diferencia entre los grupos? ¿Por qué no del todo?
4. Pista para la pregunta 3: mira cómo se crearon los barrios en la celda 2. ¿Qué información lleva la columna **barrio** sin que nadie lo haya querido?
5. El modelo se entrenó con las decisiones del pasado. ¿De quién es la culpa del sesgo: del algoritmo, de los datos o de quien decidió usar esos datos?